# Overview

This notebook drives the Version Clustering (VC) concept-drift-detection approach end-to-end - preprocessing event logs, running the clustering-based drift prediction pipeline, and evaluating the predictions against ground truth - across three different dataset setups.

**Section 1 - CDLG dataset:**
- **Predict:** runs `clt_app.processing_pipeline_cdlg` on `datasets/data_cdlg_small` with chosen hyperparameters (`n_windows`, `min_cluster_size`, `pen`, `threshold`) to detect changepoints in each log.
- **Evaluate:** runs `epp.evaluation_pipeline` and `cd_eval.main` to score the predictions against ground truth, producing precision/recall/F1 classification reports at several lag tolerances (0.01, 0.025, 0.05), broken down by drift type and noise level.
- **Sensitivity analysis:** loops predict+evaluate over combinations of `n_windows`, `min_cluster_size`, and `pen`, saving each combination's results to its own `results_sensitivity/...` folder, to see how results vary with hyperparameters.

**Section 2 - CDRIFT dataset** (Bose, Ceravolo, Ostovar; from the paper *"An Experimental Evaluation of Process Concept Drift Detection"*):
- **Preprocessing - ground truth:** reads `algorithm_results_cdrift_original.csv` and, for each dataset, extracts the actual changepoints per log and reformats them into a `drift_info.csv` per dataset, in the format the VC pipeline expects.
- **Preprocessing - Bose fix:** the Bose log only has ~1200 case IDs despite having far more trace variants, so this step reassigns a unique case ID per (case, variant) pair to expand it to ~6k distinct cases, then re-exports it as `bose_log.xes`.
- **Sensitivity analysis:** same idea as Section 1, looping `processing_pipeline_general` + `evaluation_pipeline` over hyperparameter combinations for each of the three datasets.
- **Predict (Ceravolo / Bose / Ostovar):** runs `clt_app.processing_pipeline_general` separately for each dataset using hyperparameters chosen for that dataset.
- **Evaluate:** runs `epp.evaluation_pipeline` to produce classification reports for the predictions.

**Section 3 - Real-life dataset:**
- **Predict only:** runs `clt_app.processing_pipeline_general` on `datasets/real`, which has no ground truth, so only drift predictions are produced (no evaluation step).

# 1: Steps to run Version Clustering for the CDLG dataset

## Predict

In [1]:
import utilities.cluster_approach as clt_app
import numpy as np
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

log_dir = "datasets/data_cdlg_small" 
n_windows = 125
min_cluster_size= 7
pen= 1.5
threshold= 0.1 

log_files, drift_info, window_info = clt_app.processing_pipeline_cdlg(log_dir,n_windows,min_cluster_size,threshold,pen)

Preprocessing Event Logs: 56 Event Log [00:48,  1.15 Event Log/s]
Finishing preprocessing of Event Logs: 100%|██████████████████████████████████████████| 56/56 [00:00<00:00, 119.00it/s]


Event logs actually processed:  56
Event logs with errors, no processed:  0


## Evaluate

In [2]:
import utilities.cluster_approach_eval as epp
import utilities.evaluate_scdd_4d as cd_eval
import warnings
import multiprocessing
import os

if __name__ == "__main__":
    # Suppress all warnings
    warnings.filterwarnings('ignore')
    
    # Parameters 
    log_dir = "datasets/data_cdlg_small" 
    eval_dir = "evaluation_results/cdlg_results"
    n_windows = 125 # Must be same as in the prediction

    # Evaluate the results
    p = epp.evaluation_pipeline(log_dir, eval_dir, n_windows)

    # Creates the report of the results
    num_cores = 127
    cd_eval.main(log_dir,eval_dir, num_cores)

Evaluating predictions: 100%|████████████████████████████████████████████████████████| 56/56 [00:01<00:00, 49.22 log/s]


Classification report is saved at: evaluation_results/cdlg_results/classification_report_general_0.05_lag.png
Classification report is saved at: evaluation_results/cdlg_results/classification_report_general_0.01_lag.png
Classification report is saved at: evaluation_results/cdlg_results/classification_report_general_0.025_lag.png
Lag in progress: 0.05
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% completed.
datasets/data_cdlg_small/drift_info.csv
Progress drift-level calculations: 100.00% completed.
Time taken for LAG=0.05: 6.61 seconds
Elapsed time for lag 0.05: 6.61 seconds

Lag in progress: 0.01
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% completed.
datasets/data_cdlg_small/drift_info.csv
Progress drift-level calculations: 100.00% completed.
Time taken for LAG=0.01: 6.24 seconds
Elapsed time for lag 0.01: 6.24 seconds

Lag in progress: 0.025
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% com

## Sensitivity analysis: predict and evaluate for different parameters

In [3]:
import itertools
import warnings
import multiprocessing
import os

import utilities.cluster_approach_eval as epp
import utilities.evaluate_scdd_4d as cd_eval
import utilities.cluster_approach as clt_app

if __name__ == "__main__":
    
    # Suppress all warnings
    warnings.filterwarnings('ignore')
    
    log_dir = "datasets/data_cdlg_small" 
    threshold= 0.5
    num_cores = 127
    
    n_windows = [75]
    min_cluster_size= [5]
    pen=[0.5,1.5]
    
    # Loop through all combinations
    for nw, mcs, p in itertools.product(n_windows, min_cluster_size, pen):
        print(f"windows={nw}, cluster_min={mcs}, pen={p}")
        output_dir = f"results_sensitivity/res_windows_{nw}_clusters_{mcs}_pen_{p}"
        
        clt_app.processing_pipeline_cdlg(log_dir,nw,mcs,threshold,p)
        epp.evaluation_pipeline(log_dir, output_dir, nw)
        cd_eval.main(log_dir,output_dir, num_cores)

windows=75, cluster_min=5, pen=0.5


Preprocessing Event Logs: 56 Event Log [01:03,  1.14s/ Event Log]
Finishing preprocessing of Event Logs: 100%|██████████████████████████████████████████| 56/56 [00:00<00:00, 135.24it/s]


Event logs actually processed:  56
Event logs with errors, no processed:  0


Evaluating predictions: 100%|████████████████████████████████████████████████████████| 56/56 [00:00<00:00, 63.90 log/s]


Classification report is saved at: results_sensitivity/res_windows_75_clusters_5_pen_0.5/classification_report_general_0.05_lag.png
Classification report is saved at: results_sensitivity/res_windows_75_clusters_5_pen_0.5/classification_report_general_0.01_lag.png
Classification report is saved at: results_sensitivity/res_windows_75_clusters_5_pen_0.5/classification_report_general_0.025_lag.png
Lag in progress: 0.05
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% completed.
datasets/data_cdlg_small/drift_info.csv
Progress drift-level calculations: 100.00% completed.
Time taken for LAG=0.05: 8.71 seconds
Elapsed time for lag 0.05: 8.71 seconds

Lag in progress: 0.01
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% completed.
datasets/data_cdlg_small/drift_info.csv
Progress drift-level calculations: 100.00% completed.
Time taken for LAG=0.01: 8.94 seconds
Elapsed time for lag 0.01: 8.94 seconds

Lag in progress: 0.025
datasets/data_

Preprocessing Event Logs: 56 Event Log [00:44,  1.27 Event Log/s]
Finishing preprocessing of Event Logs: 100%|██████████████████████████████████████████| 56/56 [00:00<00:00, 103.83it/s]


Event logs actually processed:  56
Event logs with errors, no processed:  0


Evaluating predictions: 100%|████████████████████████████████████████████████████████| 56/56 [00:00<00:00, 63.49 log/s]


Classification report is saved at: results_sensitivity/res_windows_75_clusters_5_pen_1.5/classification_report_general_0.05_lag.png
Classification report is saved at: results_sensitivity/res_windows_75_clusters_5_pen_1.5/classification_report_general_0.01_lag.png
Classification report is saved at: results_sensitivity/res_windows_75_clusters_5_pen_1.5/classification_report_general_0.025_lag.png
Lag in progress: 0.05
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% completed.
datasets/data_cdlg_small/drift_info.csv
Progress drift-level calculations: 100.00% completed.
Time taken for LAG=0.05: 5.72 seconds
Elapsed time for lag 0.05: 5.72 seconds

Lag in progress: 0.01
datasets/data_cdlg_small/drift_info.csv
Progress change point level: 100.00% completed.
datasets/data_cdlg_small/drift_info.csv
Progress drift-level calculations: 100.00% completed.
Time taken for LAG=0.01: 5.63 seconds
Elapsed time for lag 0.01: 5.63 seconds

Lag in progress: 0.025
datasets/data_

# 2: Steps to run Version Clustering for the CDRIFT dataset
#### From paper: An Experimental Evaluation of Process Concept Drift Detection

## Preprocessing step: Get ground truth for Bose, Ceravolo and Ostovar datasets

In [4]:
import pandas as pd
import ast

df_dir = "algorithm_results_cdrift_original.csv"
datasets_path = "datasets/cdrift"
df_full = pd.read_csv(df_dir)


datasets=["Bose","Ceravolo","Ostovar"]

for dataset in datasets:
    print(dataset)
    df_reduced = df_full[df_full["Log Source"] == dataset]
    df_reduced = df_reduced.drop_duplicates(subset="Log", keep="first")[["Log","Actual Changepoints for Log"]]
    df_reduced.columns = ["log_name", "drift_traces_index"]
    df_reduced["log_name"] = (df_reduced["log_name"] + ".xes").astype(str)

    df_reduced["drift_traces_index"] = df_reduced["drift_traces_index"].apply(lambda x: list(map(int, ast.literal_eval(x))))
    df_reduced = df_reduced.explode(['drift_traces_index']).reset_index(drop=True)
    df_reduced['drift_traces_index'] = df_reduced['drift_traces_index'].apply(lambda x: [x])

    df_reduced['change_trace_index'] = df_reduced['drift_traces_index']
    df_reduced["drift_type"] = "sudden"
    df_reduced["change_type"] = "sudden"


    # Add drift label (drift_1, drift_2, etc.)
    df_reduced['drift_or_noise_id'] = (
        df_reduced.groupby('log_name').cumcount() + 1
    ).apply(lambda x: f'drift_{x}')

    file_name= f"{datasets_path}/{dataset}/drift_info.csv"
    df_reduced.to_csv(file_name, index=False)

Bose
Ceravolo
Ostovar


### Preprocess Bose dataset
Problem: Bose dataset have only 1200 case ids, but many variants. This preprocessing is just assigning different case ids to all the variants. To have the total of almost 6k cases

In [5]:
import pm4py
from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter

log_dir = "datasets/cdrift/Bose/bose_log.xes"
df = pm4py.read_xes(log_dir)  # this returns a pandas DataFrame

df = df.sort_values(by=[ "time:timestamp"]).reset_index(drop=True)
df["id_per_case"] = df.groupby(["case:concept:name", "case:variant"]).ngroup() + 1

df["case:concept:name"] = df["id_per_case"]

# Ensure your timestamp column is in datetime format
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Convert DataFrame to EventLog
log = log_converter.apply(df)

from pm4py.objects.log.exporter.xes import exporter as xes_exporter

# Save bose log
xes_exporter.apply(log, "bose_log.xes")

exporting log, completed traces :: 100%|█████████████████████████████████████████| 5986/5986 [00:05<00:00, 1183.24it/s]


## Sensitivity analysis: predict and evaluate for different parameters

In [ ]:
import itertools
import warnings
import multiprocessing
import os

import utilities.cluster_approach_eval as epp
import utilities.evaluate_scdd_4d as cd_eval
import utilities.cluster_approach as clt_app

if __name__ == "__main__":
    
    # Suppress all warnings
    warnings.filterwarnings('ignore')

    for dataset_name in ["Ceravolo","Bose","Ostovar"]:
    
        log_dirs = [f"datasets/cdrift/{dataset_name}"]
        threshold= 0.1
        num_cores = 127
        
        n_windows = [125]
        min_cluster_size= [5,]
        pen=[0.5,1.5]
        
        for log_dir in log_dirs:
        
            # Loop through all combinations
            for nw, mcs, p in itertools.product(n_windows, min_cluster_size, pen):
                print(f"windows={nw}, cluster_min={mcs}, pen={p}")
                output_dir = f"results_{dataset_name}/res_windows_{nw}_clusters_{mcs}_pen_{p}"
                
                clt_app.processing_pipeline_general(log_dir,nw,mcs,threshold,p)
                epp.evaluation_pipeline(log_dir, output_dir, nw)
                #cd_eval.main(output_dir, num_cores)

windows=125, cluster_min=5, pen=0.5


Preprocessing Event Logs: 225 Event Log [01:04,  3.50 Event Log/s]
Finishing preprocessing of Event Logs: 100%|████████████████████████████████████████| 225/225 [00:00<00:00, 411.76it/s]


Event logs actually processed:  150
Event logs with errors, no processed:  75


Evaluating predictions: 100%|██████████████████████████████████████████████████████| 150/150 [00:03<00:00, 49.77 log/s]


Classification report is saved at: results_Ceravolo/res_windows_125_clusters_5_pen_0.5/classification_report_general_0.05_lag.png
Classification report is saved at: results_Ceravolo/res_windows_125_clusters_5_pen_0.5/classification_report_general_0.01_lag.png
Classification report is saved at: results_Ceravolo/res_windows_125_clusters_5_pen_0.5/classification_report_general_0.025_lag.png
windows=125, cluster_min=5, pen=1.5


Preprocessing Event Logs: 225 Event Log [01:07,  3.35 Event Log/s]
Finishing preprocessing of Event Logs: 100%|████████████████████████████████████████| 225/225 [00:00<00:00, 410.17it/s]


... (12 outputs left out)

Event logs actually processed:  75
Event logs with errors, no processed:  0


Evaluating predictions: 100%|████████████████████████████████████████████████████████| 75/75 [00:02<00:00, 32.94 log/s]


Classification report is saved at: results_Ostovar/res_windows_125_clusters_5_pen_0.5/classification_report_general_0.05_lag.png
Classification report is saved at: results_Ostovar/res_windows_125_clusters_5_pen_0.5/classification_report_general_0.01_lag.png
Classification report is saved at: results_Ostovar/res_windows_125_clusters_5_pen_0.5/classification_report_general_0.025_lag.png
windows=125, cluster_min=5, pen=1.5


Preprocessing Event Logs: 47 Event Log [01:49,  2.75s/ Event Log]

## Predict Ceravolo

In [2]:
import utilities.cluster_approach as clt_app
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

log_dir = "datasets/cdrift/Ceravolo"
n_windows = 125
min_cluster_size= 5
threshold= 0.3
pen= 3 

log_files, drift_info, window_info = clt_app.processing_pipeline_general(log_dir,n_windows,min_cluster_size,threshold,pen)

Preprocessing Event Logs: 225 Event Log [00:33,  6.77 Event Log/s]
Finishing preprocessing of Event Logs: 100%|█████████████████████████████████| 225/225 [00:00<00:00, 571.11it/s]


Event logs actually processed:  150
Event logs with errors, no processed:  75


## Predict Bose

In [2]:
import utilities.cluster_approach as clt_app
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')
log_dir = "datasets/cdrift/Bose"
n_windows = 75
min_cluster_size= 5
threshold= 0.3
pen= 3

log_files, drift_info, window_info = clt_app.processing_pipeline_general(log_dir,n_windows,min_cluster_size,threshold,pen)

Preprocessing Event Logs: 0 Event Log [00:00, ? Event Log/s]/usr/local/lib/python3.12/site-packages/pm4py/algo/discovery/dfg/adapters/pandas/df_statistics.py:92: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[start_timestamp_key] = df[timestamp_key]
/usr/local/lib/python3.12/site-packages/pm4py/algo/discovery/dfg/adapters/pandas/df_statistics.py:92: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[start_timestamp_key] = df[timestamp_key]
/usr/local/lib/python3.12/site-packages/pm4py/algo/discovery/dfg/ada

Event logs actually processed:  1
Event logs with errors, no processed:  0


# Predict Ostovar

In [1]:
import utilities.cluster_approach as clt_app
import warnings
# Suppress all warnings
warnings.filterwarnings('ignore')

log_dir = "datasets/cdrift/Ostovar"
n_windows = 75
min_cluster_size= 7
pen= 3
threshold= 0.3

log_files, drift_info, window_info = clt_app.processing_pipeline_general(log_dir,n_windows,min_cluster_size,threshold,pen)

Preprocessing Event Logs: 75 Event Log [01:05,  1.15 Event Log/s]
Finishing preprocessing of Event Logs: 100%|███████████████████████████████████| 75/75 [00:00<00:00, 398.86it/s]


Event logs actually processed:  75
Event logs with errors, no processed:  0


## Evaluate

In [ ]:
import utilities.cluster_approach_eval as epp
import utilities.evaluate_scdd_4d as cd_eval
import warnings
import multiprocessing
import os

if __name__ == "__main__":
    # Suppress all warnings
    warnings.filterwarnings('ignore')
    
    # Parameters 
    test_dir = "datasets/cdrift/Ceravolo"
    output_dir = "evaluation_results/cdrift_results"
    n_windows = 125
    
    p = epp.evaluation_pipeline(test_dir, output_dir, n_windows)


# 3: Steps to run Version Clustering for real life dataset

It predicts without ground truth

## Predict

In [ ]:
import utilities.cluster_approach as clt_app
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

log_dir = "datasets/real"
n_windows = 100
min_cluster_size= 5
threshold= 0.5
pen= 2.5

log_files, drift_info, window_info = clt_app.processing_pipeline_general(log_dir,n_windows,min_cluster_size,threshold,pen)